In [ ]:
# ==================== 导入必要的库 ====================
# random: Python标准库，用于随机数生成
import random

# torch: PyTorch深度学习框架
import torch

# re: Python正则表达式库，用于文本处理
import re

# d2l: Dive into Deep Learning工具库
from d2l import torch as d2l

In [ ]:
# ==================== 读取和处理时间机器数据集 ====================

# 注册数据集到d2l数据中心
# 'time_machine': 数据集名称
# 第一个参数：下载URL
# 第二个参数：文件的SHA-1校验码，用于验证文件完整性
d2l.DATA_HUB['time_machine'] = (d2l.DATA_URL + 'timemachine.txt', 
                                '090b5e7e70c295757f55df93cb0a180b9691891a')


def read_time_machine():
    """
    加载时间机器数据集到文本行列表
    
    什么是时间机器数据集？
    - H.G. Wells的科幻小说《时间机器》的文本
    - 常用于自然语言处理和语言模型的教学示例
    
    处理步骤：
    1. 下载数据集文件
    2. 读取所有行
    3. 使用正则表达式清理文本：
       - [^A-Za-z]+: 匹配所有非字母字符（数字、标点等）
       - 替换为空格
       - 转换为小写（统一格式）
       - 去除首尾空格
    
    为什么要清理文本？
    - 简化词汇表（只保留字母）
    - 统一大小写（'The'和'the'被视为同一个词）
    - 便于后续的分词和建模
    
    返回:
        清理后的文本行列表
    """
    # 下载数据集并打开文件
    with open(d2l.download('time_machine'), 'r') as f:
        lines = f.readlines()  # 读取所有行到列表
    
    # 清理每一行文本
    # re.sub(pattern, replacement, string): 正则替换
    # [^A-Za-z]+: 匹配一个或多个非字母字符
    # strip(): 去除首尾空白
    # lower(): 转换为小写
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]


# ==================== 分词和构建词汇表 ====================

# 分词：将文本拆分成词（token）
# d2l.tokenize()会将每行文本拆分成单词列表
# 返回：列表的列表，外层列表对应每一行，内层列表对应该行的单词
tokens = d2l.tokenize(read_time_machine())

# 将所有文本行的词拼接成一个长序列（语料库corpus）
# 为什么要拼接？
# - 文本行的划分是人为的，不一定对应句子或段落
# - 语言模型需要连续的文本序列
# - 列表推导式：[token for line in tokens for token in line]
#   遍历每一行，再遍历该行的每个词
corpus = [token for line in tokens for token in line]

# 构建词汇表（Vocabulary）
# 什么是词汇表？
# - 语料库中所有唯一词的集合
# - 为每个词分配一个唯一的索引（ID）
# - 统计每个词的出现频率
# 
# d2l.Vocab会：
# 1. 统计每个词的频率
# 2. 按频率降序排序
# 3. 为每个词分配索引
vocab = d2l.Vocab(corpus)

# 查看词频最高的前10个词
# token_freqs: 列表，每个元素是(词, 频率)的元组
# 预期：最常见的词通常是'the', 'a', 'and'等功能词
vocab.token_freqs[:10]

In [ ]:
# ==================== 可视化词频分布 ====================

# 提取所有词的频率（不需要词本身，只要频率）
freqs = [freq for token, freq in vocab.token_freqs]

# 绘制词频分布图
# 
# 使用对数坐标（xscale='log', yscale='log'）
# 为什么用对数坐标？
# - 词频分布非常不均匀
# - 少数词非常常见（如'the'出现数千次）
# - 大多数词很少见（可能只出现1-2次）
# - 对数坐标能更好地展示这种幂律分布
# 
# 齐夫定律（Zipf's Law）：
# - 自然语言中的一个经验规律
# - 词频与排名成反比：frequency ∝ 1/rank
# - 在对数坐标系中表现为直线
# - 第k常见的词的频率约为最常见词的1/k
d2l.plot(freqs, xlabel='token: x', ylabel='frequency: n(x)',
         xscale='log', yscale='log')

In [ ]:
# ==================== 二元语法（Bigram） ====================
# 什么是n元语法（n-gram）？
# - 连续n个词的序列
# - Unigram（1-gram）：单个词，如'the'
# - Bigram（2-gram）：两个连续的词，如'the time'
# - Trigram（3-gram）：三个连续的词，如'the time machine'
# 
# 为什么需要n-gram？
# - 捕捉词与词之间的关系
# - 'New York'比'New'和'York'分开更有意义
# - 提高语言模型的表达能力

# 构造二元语法tokens
# zip(corpus[:-1], corpus[1:]): 将相邻的词配对
# 例如：corpus = ['the', 'time', 'machine', 'by']
#       pairs = [('the', 'time'), ('time', 'machine'), ('machine', 'by')]
# 
# [:-1]: 从第一个词到倒数第二个词
# [1:]: 从第二个词到最后一个词
# zip将它们配对
bigram_tokens = [' '.join(pair) for pair in zip(corpus[:-1], corpus[1:])]

# 为二元语法构建词汇表
bigram_vocab = d2l.Vocab(bigram_tokens)

# 查看最常见的10个二元语法及其频率
# 例如：'of the', 'in the', 'to the'等
# 返回格式：[(('of', 'the'), 频率), ...]
[(tuple(token.split()), freq) for token, freq in bigram_vocab.token_freqs[:10]]

In [ ]:
# ==================== 三元语法（Trigram） ====================

# 构造三元语法tokens
# 将三个连续的词组合在一起
# 
# zip的多参数用法：
# corpus[:-2]: 第1个词到倒数第3个词
# corpus[1:-1]: 第2个词到倒数第2个词
# corpus[2:]: 第3个词到最后一个词
# 
# 例如：corpus = ['the', 'time', 'machine', 'by', 'h']
#       triples = [('the', 'time', 'machine'), 
#                  ('time', 'machine', 'by'),
#                  ('machine', 'by', 'h')]
trigram_tokens = [' '.join(triple) for triple in zip(
    corpus[:-2], corpus[1:-1], corpus[2:])]

# 为三元语法构建词汇表
trigram_vocab = d2l.Vocab(trigram_tokens)

# 查看最常见的10个三元语法
# 三元语法能捕捉更复杂的短语和表达
# 例如：'one of the', 'it was a'等
trigram_vocab.token_freqs[:10]

In [ ]:
# ==================== 对比不同n-gram的词频分布 ====================

# 提取bigram和trigram的频率
bigram_freqs = [freq for token, freq in bigram_vocab.token_freqs]
trigram_freqs = [freq for token, freq in trigram_vocab.token_freqs]

# 在同一张图上绘制unigram、bigram、trigram的词频分布
# 
# 观察：
# 1. 都遵循齐夫定律（对数坐标上近似直线）
# 2. n越大，词汇量越大：
#    - Unigram: ~4000个唯一词
#    - Bigram: 更多（因为组合增加）
#    - Trigram: 最多
# 3. n越大，频率越分散：
#    - Unigram有很多高频词
#    - Trigram中大多数只出现1-2次
# 
# 启示：
# - 更大的n能捕捉更多上下文
# - 但也导致数据稀疏问题
# - 需要更多数据或更好的模型（如神经语言模型）
d2l.plot([freqs, bigram_freqs, trigram_freqs], xlabel='token: x',
         ylabel='frequency: n(x)', xscale='log', yscale='log',
         legend=['unigram', 'bigram', 'trigram'])

In [ ]:
# ==================== 随机采样的序列数据迭代器 ====================

def seq_data_iter_random(corpus, batch_size, num_steps):
    """
    使用随机抽样生成小批量子序列
    
    什么是随机采样？
    - 从语料库中随机选取多个起始位置
    - 从每个起始位置提取固定长度的序列
    - 不同批次间的序列不保证连续
    
    参数:
        corpus: 词索引列表（整个语料库）
        batch_size: 批次大小（每批包含多少个序列）
        num_steps: 每个序列的长度（时间步数）
    
    生成:
        X, Y: 输入和标签张量
        - X的形状: (batch_size, num_steps)
        - Y的形状: (batch_size, num_steps)
        - Y是X向后偏移1位的结果（预测下一个词）
    
    优点：
    - 简单直接
    - 每个epoch都能看到不同的序列组合
    
    缺点：
    - 批次间的序列不连续，丢失了跨批次的上下文信息
    """
    # 从随机偏移量开始对序列进行分区
    # 随机偏移0到num_steps-1位置，避免每次都从同一位置开始
    corpus = corpus[random.randint(0, num_steps - 1):]
    
    # 计算可以分成多少个长度为num_steps的子序列
    # 减1是因为需要为标签留出空间（Y=X向后偏移1）
    num_subseqs = (len(corpus) - 1) // num_steps
    
    # 生成所有子序列的起始索引
    # range(0, num_subseqs * num_steps, num_steps)
    # 例如：num_steps=5时，起始位置为[0, 5, 10, 15, ...]
    initial_indices = list(range(0, num_subseqs * num_steps, num_steps))
    
    # 随机打乱起始索引
    # 这样在随机抽样的迭代过程中，
    # 来自两个相邻的、随机的、小批量中的子序列
    # 不一定在原始序列上相邻
    random.shuffle(initial_indices)

    def data(pos):
        """
        返回从pos位置开始的长度为num_steps的序列
        
        参数:
            pos: 起始位置
        
        返回:
            长度为num_steps的子序列
        """
        return corpus[pos: pos + num_steps]

    # 计算可以产生多少个批次
    num_batches = num_subseqs // batch_size
    
    # 生成每个批次
    for i in range(0, batch_size * num_batches, batch_size):
        # 获取当前批次的起始索引
        # 每批包含batch_size个序列
        initial_indices_per_batch = initial_indices[i: i + batch_size]
        
        # 构造输入X：从每个起始位置提取num_steps个词
        X = [data(j) for j in initial_indices_per_batch]
        
        # 构造标签Y：从每个起始位置+1提取num_steps个词
        # Y[i] = X[i]的下一个词，即X向右偏移1位
        Y = [data(j + 1) for j in initial_indices_per_batch]
        
        # 转换为张量并返回
        yield torch.tensor(X), torch.tensor(Y)

In [ ]:
# ==================== 测试随机采样 ====================

# 创建一个简单的序列用于演示
# my_seq = [0, 1, 2, 3, ..., 34]
my_seq = list(range(35))

# 使用随机采样生成批次
# batch_size=2: 每批2个序列
# num_steps=5: 每个序列长度为5
# 
# 预期输出：
# - 每批有2个序列
# - 每个序列有5个元素
# - X和Y的关系：Y[i] = X[i] + 1（下一个元素）
# - 不同批次的序列起始位置是随机的、不连续的
for X, Y in seq_data_iter_random(my_seq, batch_size=2, num_steps=5):
    print('X: ', X, '\nY:', Y)

In [ ]:
# ==================== 顺序分区的序列数据迭代器 ====================

def seq_data_iter_sequential(corpus, batch_size, num_steps):
    """
    使用顺序分区生成小批量子序列
    
    什么是顺序分区？
    - 将语料库分成batch_size个连续的部分
    - 每个部分按顺序生成序列
    - 同一批次内的序列在原文中的位置不同，但批次间保持连续
    
    参数:
        corpus: 词索引列表
        batch_size: 批次大小
        num_steps: 每个序列的长度
    
    生成:
        X, Y: 输入和标签张量
    
    优点：
    - 保持了序列的连续性
    - 可以维护跨批次的隐藏状态
    - 更适合训练RNN
    
    缺点：
    - 每个epoch看到的序列顺序相同
    
    示例（batch_size=2, num_steps=5）：
    原序列: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,...]
    分为2批:
    批次1: [0,1,2,3,4]    批次2: [5,6,7,8,9]    ...
           [8,9,10,11,12]        [13,14,15,16,17] ...
    """
    # 从随机偏移量开始划分序列
    # 添加一点随机性，避免每次epoch从完全相同的位置开始
    offset = random.randint(0, num_steps)
    
    # 计算可用的tokens数量
    # 需要能被batch_size整除，以便均匀分配
    num_tokens = ((len(corpus) - offset - 1) // batch_size) * batch_size
    
    # 提取输入序列X和标签序列Y
    # Y相对于X向后偏移1位
    Xs = torch.tensor(corpus[offset: offset + num_tokens])
    Ys = torch.tensor(corpus[offset + 1: offset + 1 + num_tokens])
    
    # 重塑为(batch_size, -1)
    # 将长序列分成batch_size行
    # 每行是一个独立的序列流
    # 
    # 例如：num_tokens=24, batch_size=2
    # 原来: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]
    # 重塑后:
    # [[0,1,2,3,4,5,6,7,8,9,10,11],
    #  [12,13,14,15,16,17,18,19,20,21,22,23]]
    Xs, Ys = Xs.reshape(batch_size, -1), Ys.reshape(batch_size, -1)
    
    # 计算可以生成多少个批次
    num_batches = Xs.shape[1] // num_steps
    
    # 按顺序生成每个批次
    for i in range(0, num_steps * num_batches, num_steps):
        # 从每一行中提取num_steps列
        # 这样保证了每个批次内的序列在时间上是连续的
        X = Xs[:, i: i + num_steps]
        Y = Ys[:, i: i + num_steps]
        yield X, Y

In [ ]:
# ==================== 测试顺序分区 ====================

# 使用相同的测试序列
# my_seq = [0, 1, 2, 3, ..., 34]

# 使用顺序分区生成批次
# 
# 观察与随机采样的区别：
# 1. 批次间连续：
#    - 第1批的最后一个元素 + 1 = 第2批的第一个元素
# 2. 批次内的两个序列：
#    - 第一个序列：从序列前半部分顺序取
#    - 第二个序列：从序列后半部分顺序取
# 3. X和Y的关系依然是：Y = X + 1
for X, Y in seq_data_iter_sequential(my_seq, batch_size=2, num_steps=5):
    print('X: ', X, '\nY:', Y)

In [ ]:
# ==================== 序列数据加载器类 ====================

class SeqDataLoader:
    """
    加载序列数据的迭代器类
    
    这个类封装了序列数据加载的逻辑：
    1. 选择采样策略（随机或顺序）
    2. 加载和处理语料库
    3. 提供统一的迭代接口
    
    使用方法：
        loader = SeqDataLoader(batch_size=32, num_steps=35, 
                               use_random_iter=False, max_tokens=10000)
        for X, Y in loader:
            # 训练模型
            ...
    """
    
    def __init__(self, batch_size, num_steps, use_random_iter, max_tokens):
        """
        初始化数据加载器
        
        参数:
            batch_size: 批次大小
            num_steps: 序列长度（时间步数）
            use_random_iter: 是否使用随机采样
                - True: 随机采样（批次间不连续）
                - False: 顺序分区（批次间连续）
            max_tokens: 最大token数量（限制语料库大小）
        """
        # 根据use_random_iter选择迭代函数
        if use_random_iter:
            # 使用随机采样
            self.data_iter_fn = d2l.seq_data_iter_random
        else:
            # 使用顺序分区
            self.data_iter_fn = d2l.seq_data_iter_sequential
        
        # 加载时间机器语料库和词汇表
        # load_corpus_time_machine会：
        # 1. 读取并清理文本
        # 2. 分词
        # 3. 构建词汇表
        # 4. 将文本转换为词索引序列
        self.corpus, self.vocab = d2l.load_corpus_time_machine(max_tokens)
        
        # 保存参数
        self.batch_size, self.num_steps = batch_size, num_steps

    def __iter__(self):
        """
        使对象可迭代
        
        返回数据迭代器，用于for循环
        """
        return self.data_iter_fn(self.corpus, self.batch_size, self.num_steps)

In [ ]:
# ==================== 加载时间机器数据集的便捷函数 ====================

def load_data_time_machine(batch_size, num_steps,
                           use_random_iter=False, max_tokens=10000):
    """
    返回时光机器数据集的迭代器和词汇表
    
    这是一个便捷函数，简化了数据加载过程
    
    参数:
        batch_size: 批次大小
        num_steps: 序列长度
        use_random_iter: 是否使用随机采样，默认False（使用顺序分区）
        max_tokens: 最大token数量，默认10000
    
    返回:
        data_iter: 数据迭代器，可用于for循环
        vocab: 词汇表对象，包含：
            - token到索引的映射
            - 索引到token的映射
            - 词频统计
    
    使用示例:
        train_iter, vocab = load_data_time_machine(32, 35)
        for X, Y in train_iter:
            # X: (32, 35) - 批次大小32，序列长度35
            # Y: (32, 35) - 对应的标签
            ...
    """
    # 创建数据迭代器
    data_iter = SeqDataLoader(
        batch_size, num_steps, use_random_iter, max_tokens)
    
    # 返回迭代器和词汇表
    # data_iter.vocab: 访问SeqDataLoader内部的词汇表
    return data_iter, data_iter.vocab